# RAID v6: Sentence Influence Matrices

## Overview
This notebook identifies **which specific sentences** carry long-range influence in a document, and how far that influence extends. Instead of asking "does more context help?" (v5), we ask "which specific prior sentences help predict each target sentence?"

## Method: Leave-One-Sentence-Out
For each target sentence (in the second half of the document):
1. **Baseline**: Compute perplexity with ALL prior sentences as context
2. **Drop sentence i**: Compute perplexity with sentence i removed from context
3. **Influence(i → target)** = log(ppl_without_i / ppl_with_i)
   - Positive = sentence i helped predict the target (removing it hurts prediction)
   - Zero = sentence i was irrelevant
   - Negative = sentence i was misleading (removing it helps)

This produces an **influence matrix** (n_sentences × n_sentences) per document.

## What This Reveals
- **Influence decay curve**: aggregate influence by sentence distance
- **Hotspots**: specific sentence pairs with strong long-range influence (e.g., character introductions affecting references 20 sentences later)
- **Sentence lifespans**: how far forward each sentence's influence extends
- **Human vs AI comparison**: do AI texts have denser long-range connections?

## Key Insight
Individual sentences do NOT decay smoothly — the smooth aggregate decay curve is an average over spiky, content-driven individual curves. Long-range influence is about **selective retrieval** of specific information, not uniform activation decay.

## Setup
Install dependencies. Requires GPU (recommended: H100 or L4 for speed).

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import json, math, time, gc, os, re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Imports OK")

## Configuration
- `N_DOCS`: Number of documents to analyze (O(n²) per doc, so keep manageable)
- `MAX_SENTENCES`: Cap on sentences per document
- Both human and AI documents are sampled, stratified across genres

In [ ]:
IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DATA = Path("/content/drive/MyDrive/LRTIA/Data/raid_sampled")
    DRIVE_RESULTS = Path("/content/drive/MyDrive/LRTIA/Results/RAID_v6")
    if (DRIVE_DATA / "raid_corpus.jsonl").exists():
        DATA_DIR = DRIVE_DATA
    else:
        LOCAL_DATA = Path("/content/data/raid_sampled")
        if not (LOCAL_DATA / "raid_corpus.jsonl").exists():
            LOCAL_DATA.mkdir(parents=True, exist_ok=True)
            from google.colab import files
            uploaded = files.upload()
            for fname in uploaded:
                with open(LOCAL_DATA / fname, 'wb') as f:
                    f.write(uploaded[fname])
        DATA_DIR = LOCAL_DATA
    BASE_DIR = DRIVE_RESULTS
    BASE_DIR.mkdir(parents=True, exist_ok=True)
else:
    BASE_DIR = Path("../results/raid_v6")
    DATA_DIR = Path("../data/raid_sampled")
    BASE_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"BASE_DIR: {BASE_DIR}")

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

# Subsample for speed — influence matrices are O(n^2) in sentences
N_DOCS = 50  # docs to analyze (human only)
MAX_SENTENCES = 30  # max sentences per doc to keep tractable
RANDOM_SEED = 42

DOMAINS = ['abstracts', 'books', 'news', 'poetry', 'recipes', 'reddit', 'reviews', 'wiki']

print(f"N docs: {N_DOCS}")
print(f"Max sentences per doc: {MAX_SENTENCES}")

## Load Corpus
Load RAID dataset, sample human AND AI documents spread across genres for comparison.

In [ ]:
corpus_all = []
with open(DATA_DIR / "raid_corpus.jsonl") as f:
    for line in f:
        corpus_all.append(json.loads(line))

rng = np.random.RandomState(RANDOM_SEED)

# Sample human AND AI docs, spread across genres
corpus = []
for pop in ['human', 'ai']:
    pop_docs = [d for d in corpus_all if d['population'] == pop and len(d['text'].split()) >= 200]
    for domain in DOMAINS:
        pool = [d for d in pop_docs if d['domain'] == domain]
        n = min(len(pool), N_DOCS // len(DOMAINS) + 1)
        corpus.extend(rng.choice(pool, size=n, replace=False))

print(f"Selected {len(corpus)} documents")
print(f"\nBy population:")
for pop in ['human', 'ai']:
    n = sum(1 for c in corpus if c['population'] == pop)
    print(f"  {pop}: {n}")
print(f"\nBy domain:")
for d in DOMAINS:
    nh = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'human')
    na = sum(1 for c in corpus if c['domain'] == d and c['population'] == 'ai')
    print(f"  {d}: human={nh}, ai={na}")

## Load Model
Mistral-7B, 4-bit quantized. Same measuring instrument as v5.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("Model loaded")

## Core Functions

**`split_into_sentences(text)`** — Sentence boundary detection.

**`compute_ppl_on_target(context_ids, target_ids)`** — Perplexity of target given context.

**`compute_influence_matrix(doc)`** — The main computation. For each target sentence t:
  - Compute baseline perplexity with full context (all sentences before t)
  - For each prior sentence i: compute perplexity with sentence i dropped
  - Influence[i,t] = log(ppl_dropped / ppl_baseline)

This is O(n²) in the number of sentences — each target requires n forward passes (one baseline + one per dropped sentence).

In [ ]:
def split_into_sentences(text):
    """Split text into sentences, return list of strings."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if len(s.strip().split()) >= 4]


@torch.no_grad()
def compute_ppl_on_target(context_token_ids, target_token_ids):
    """Perplexity of target tokens given context tokens."""
    full_ids = context_token_ids + target_token_ids
    input_ids = torch.tensor([full_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    target_start = len(context_token_ids)
    total_loss = 0.0
    count = 0
    for i in range(target_start, len(full_ids) - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[full_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')


def compute_influence_matrix(doc):
    """Compute sentence-by-sentence influence matrix.
    matrix[i, t] = influence of sentence i on target sentence t
    = log(ppl_without_i / ppl_with_i)
    positive = sentence i helped predict sentence t
    """
    sentences = split_into_sentences(doc['text'])
    if len(sentences) < 4:
        return None, None

    if len(sentences) > MAX_SENTENCES:
        sentences = sentences[:MAX_SENTENCES]

    n = len(sentences)
    sent_ids = [tokenizer.encode(s, add_special_tokens=False) for s in sentences]

    matrix = np.full((n, n), np.nan)

    for t in range(2, n):
        target_ids = sent_ids[t]
        if len(target_ids) < 3:
            continue

        full_context_ids = []
        for k in range(t):
            full_context_ids.extend(sent_ids[k])

        ppl_full = compute_ppl_on_target(full_context_ids, target_ids)
        if math.isinf(ppl_full) or ppl_full <= 0:
            continue

        for i in range(t):
            dropped_context_ids = []
            for k in range(t):
                if k != i:
                    dropped_context_ids.extend(sent_ids[k])

            if len(dropped_context_ids) < 2:
                continue

            ppl_dropped = compute_ppl_on_target(dropped_context_ids, target_ids)
            if math.isinf(ppl_dropped) or ppl_dropped <= 0:
                continue

            matrix[i, t] = math.log(ppl_dropped / ppl_full)

    return sentences, matrix


print("Functions defined")

## Compute Influence Matrices
Run the leave-one-out analysis for all documents. Saves matrices and metadata (including sentence texts) for later analysis.

This is the expensive step. ~50 docs × 30 sentences × ~15 forward passes per target = ~10K-20K forward passes total.

In [ ]:
results_path = BASE_DIR / "influence_matrices_v2.npz"
meta_path = BASE_DIR / "influence_meta_v2.json"

if results_path.exists() and meta_path.exists():
    loaded = np.load(results_path, allow_pickle=True)
    all_matrices = list(loaded['matrices'])
    with open(meta_path) as f:
        all_meta = json.load(f)
    print(f"Loaded {len(all_matrices)} influence matrices")
else:
    all_matrices = []
    all_meta = []

    for doc in tqdm(corpus, desc="Computing influence matrices"):
        sentences, matrix = compute_influence_matrix(doc)
        if matrix is None:
            continue
        all_matrices.append(matrix)
        all_meta.append({
            'doc_id': doc['doc_id'],
            'domain': doc['domain'],
            'population': doc['population'],
            'n_sentences': len(sentences),
            'sentences': sentences,
        })

    np.savez(results_path, matrices=np.array(all_matrices, dtype=object))
    with open(meta_path, 'w') as f:
        json.dump(all_meta, f)
    print(f"Computed {len(all_matrices)} matrices, saved to {BASE_DIR}")

print(f"Documents with matrices: {len(all_matrices)}")
print(f"  Human: {sum(1 for m in all_meta if m['population'] == 'human')}")
print(f"  AI: {sum(1 for m in all_meta if m['population'] == 'ai')}")

## Analysis: Human vs AI
Aggregate influence values by sentence distance, separately for human and AI text.

Key comparisons:
- **Decay curves**: Same shape for human vs AI, or different?
- **Hotspot density**: Does AI have more long-range hotspots?
- **Monotonicity**: Are individual sentence influence curves smooth or spiky?
- **Percentage positive**: At each distance, what fraction of sentences have positive influence?

In [ ]:
# ── Analysis: Aggregate influence by sentence distance, HUMAN vs AI ──

by_dist_pop = {'human': {}, 'ai': {}}

for doc_idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    meta = all_meta[doc_idx]
    pop = meta['population']

    for i in range(n):
        for t in range(i+1, n):
            val = matrix[i, t]
            if not np.isnan(val):
                d = t - i
                if d not in by_dist_pop[pop]:
                    by_dist_pop[pop][d] = []
                by_dist_pop[pop][d].append(val)

print("INFLUENCE BY SENTENCE DISTANCE: HUMAN vs AI")
print(f"{'Dist':>5} {'H mean':>10} {'A mean':>10} {'H-A':>10} {'H %+':>8} {'A %+':>8} {'H n':>8} {'A n':>8}")
print("-" * 75)
for d in range(1, 25):
    h_vals = np.array(by_dist_pop['human'].get(d, []))
    a_vals = np.array(by_dist_pop['ai'].get(d, []))
    if len(h_vals) < 3 or len(a_vals) < 3:
        continue
    h_pct = 100 * np.mean(h_vals > 0)
    a_pct = 100 * np.mean(a_vals > 0)
    print(f"{d:>5} {h_vals.mean():>10.4f} {a_vals.mean():>10.4f} {h_vals.mean()-a_vals.mean():>+10.4f} {h_pct:>7.1f}% {a_pct:>7.1f}% {len(h_vals):>8} {len(a_vals):>8}")

print("\n\nHOTSPOT ANALYSIS: Human vs AI")
for threshold in [0.05, 0.1, 0.2]:
    for min_dist in [5, 10]:
        for pop in ['human', 'ai']:
            count = 0
            total = 0
            for d in sorted(by_dist_pop[pop].keys()):
                if d >= min_dist:
                    vals = np.array(by_dist_pop[pop][d])
                    count += np.sum(vals > threshold)
                    total += len(vals)
            pct = 100 * count / total if total > 0 else 0
            print(f"  {pop:<6} influence>{threshold}, dist>={min_dist}: {count}/{total} = {pct:.1f}%")
        print()

print("\nMONOTONICITY: Human vs AI")
for pop in ['human', 'ai']:
    mono_scores = []
    for doc_idx, matrix in enumerate(all_matrices):
        if all_meta[doc_idx]['population'] != pop:
            continue
        n = matrix.shape[0]
        for i in range(n - 5):
            vals = [matrix[i, t] for t in range(i+1, n) if not np.isnan(matrix[i, t])]
            if len(vals) < 8:
                continue
            decreasing = sum(1 for j in range(len(vals)-1) if vals[j] > vals[j+1])
            mono_scores.append(decreasing / (len(vals) - 1))
    mono_scores = np.array(mono_scores)
    print(f"  {pop:<6}: mean monotonicity = {mono_scores.mean():.3f} (n={len(mono_scores)})")

## Visualization
- **Panel A**: Mean influence by sentence distance, human vs AI (with error bars)
- **Panel B**: Distribution of influence values at distance 1 vs distance 10
- **Panel C/D**: Example influence matrix heatmaps for human and AI text

The heatmaps show the full influence structure of a single document. Look for:
- Strong diagonal (adjacent sentences matter most)
- Off-diagonal hotspots (specific long-range dependencies)
- Differences in hotspot density between human and AI

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Mean influence by sentence distance — Human vs AI
ax = axes[0, 0]
for pop, color, label in [('human', '#3498db', 'Human'), ('ai', '#e74c3c', 'AI')]:
    dists = sorted(by_dist_pop[pop].keys())
    dists = [d for d in dists if len(by_dist_pop[pop][d]) >= 5]
    means = [np.mean(by_dist_pop[pop][d]) for d in dists]
    sems = [np.std(by_dist_pop[pop][d]) / np.sqrt(len(by_dist_pop[pop][d])) for d in dists]
    ax.errorbar(dists, means, yerr=sems, fmt='o-', color=color, linewidth=2, markersize=5, capsize=2, label=label)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel('Mean Influence\nlog(ppl_without / ppl_with)')
ax.set_title('A. Influence Decay: Human vs AI', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)

# B: Distribution comparison at distance 1 vs distance 10
ax = axes[0, 1]
for d_plot, alpha in [(1, 0.6), (10, 0.4)]:
    for pop, color, label in [('human', '#3498db', 'Human'), ('ai', '#e74c3c', 'AI')]:
        if d_plot in by_dist_pop[pop]:
            vals = by_dist_pop[pop][d_plot]
            ax.hist(vals, bins=25, alpha=alpha, color=color,
                    label=f'{label} d={d_plot} (n={len(vals)})', density=True)
ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Influence (log ratio)')
ax.set_ylabel('Density')
ax.set_title('B. Influence Distributions: d=1 vs d=10', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.2)

# C: Example heatmaps — one human, one AI
for plot_idx, pop in enumerate(['human', 'ai']):
    ax = axes[1, plot_idx]
    # Find a good example for this population
    best_idx = None
    best_n = 0
    for doc_idx, meta in enumerate(all_meta):
        matching = [d for d in corpus if d['doc_id'] == meta['doc_id']]
        if matching and matching[0]['population'] == pop:
            if meta['n_sentences'] > best_n:
                best_n = meta['n_sentences']
                best_idx = doc_idx
    if best_idx is not None:
        matrix = all_matrices[best_idx]
        meta = all_meta[best_idx]
        masked = np.where(np.isnan(matrix), 0, matrix)
        im = ax.imshow(masked, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
        ax.set_xlabel('Target Sentence')
        ax.set_ylabel('Source Sentence')
        ax.set_title(f'C{"" if pop == "human" else "d"}. {pop.capitalize()} Example\n({meta["domain"]}, {meta["n_sentences"]} sents)',
                     fontweight='bold', fontsize=11)
        plt.colorbar(im, ax=ax, label='Influence', shrink=0.8)

plt.suptitle('Sentence Influence: Human vs AI Text',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_influence_human_vs_ai.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Mean influence by sentence distance
ax = axes[0, 0]
dists = sorted(influence_by_distance.keys())
means = [np.mean(influence_by_distance[d]) for d in dists]
sems = [np.std(influence_by_distance[d]) / np.sqrt(len(influence_by_distance[d])) for d in dists]
ax.errorbar(dists, means, yerr=sems, fmt='o-', color='#3498db', linewidth=2, markersize=6, capsize=3)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Sentence Distance')
ax.set_ylabel('Mean Influence\nlog(ppl_without / ppl_with)')
ax.set_title('A. Influence Decay by Sentence Distance', fontweight='bold')
ax.grid(True, alpha=0.2)

# B: Distribution of influence values at different distances
ax = axes[0, 1]
for d, color in [(1, '#e74c3c'), (3, '#f39c12'), (6, '#27ae60'), (10, '#3498db')]:
    if d in influence_by_distance:
        vals = influence_by_distance[d]
        ax.hist(vals, bins=30, alpha=0.4, color=color, label=f'dist={d} (n={len(vals)})', density=True)
ax.axvline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Influence (log ratio)')
ax.set_ylabel('Density')
ax.set_title('B. Influence Distribution at Different Distances', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# C: Example influence matrix (heatmap)
ax = axes[1, 0]
# Pick a doc with a good number of sentences
best_idx = max(range(len(all_matrices)), key=lambda i: all_matrices[i].shape[0])
matrix = all_matrices[best_idx]
meta = all_meta[best_idx]
# Mask the upper triangle and diagonal (only show i < t)
masked = np.where(np.isnan(matrix), 0, matrix)
im = ax.imshow(masked, cmap='RdBu_r', aspect='auto', vmin=-0.3, vmax=0.3)
ax.set_xlabel('Target Sentence')
ax.set_ylabel('Source Sentence')
ax.set_title(f'C. Example Influence Matrix\n({meta["domain"]}, {meta["n_sentences"]} sentences)', fontweight='bold', fontsize=11)
plt.colorbar(im, ax=ax, label='Influence', shrink=0.8)

# D: Variance of influence — are some sentences more influential than others?
ax = axes[1, 1]
# For each source sentence, compute its mean influence across all targets
src_influence = {}  # src_position -> list of mean influences across docs
for idx, matrix in enumerate(all_matrices):
    n = matrix.shape[0]
    for i in range(n):
        row_vals = matrix[i, i+1:]
        row_vals = row_vals[~np.isnan(row_vals)]
        if len(row_vals) >= 2:
            if i not in src_influence:
                src_influence[i] = []
            src_influence[i].append(np.mean(row_vals))

positions = sorted([p for p in src_influence.keys() if len(src_influence[p]) >= 5])
means = [np.mean(src_influence[p]) for p in positions]
sds = [np.std(src_influence[p]) for p in positions]

ax.bar(positions, means, yerr=sds, color='#3498db', alpha=0.7, capsize=2)
ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Source Sentence Position')
ax.set_ylabel('Mean Forward Influence')
ax.set_title('D. Which Sentences Are Most Influential?', fontweight='bold')
ax.grid(True, alpha=0.2, axis='y')

plt.suptitle('Sentence Influence Analysis: How Far Does Each Sentence Reach?',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig1_influence.png', dpi=150, bbox_inches='tight')
plt.show()